# Как это считается: соберите решатель сами

Здесь нет картинок — только числа и код. Каждый шаг расчёта, который делает большой ноутбук с анимацией, разложен на части, чтобы можно было повторить руками и проверить себя.

**Что решаем.** Телеграфные уравнения (методичка, глава 2):

```
−∂U/∂x = L′·∂I/∂t + R′·I
−∂I/∂x = C′·∂U/∂t + G′·U
```

Параметры берём как в лабораторной: $Z_0 = 50$ Ом, $v = 2\cdot10^8$ м/с, линия 143 м (пять длин волн на 7 МГц). Отсюда сразу погонные параметры:

$$C' = \frac{1}{Z_0 v} = 100\ \text{пФ/м}, \qquad L' = \frac{Z_0}{v} = 250\ \text{нГн/м}$$


## Шаг 1. Линия превращается в лестницу

Непрерывную линию режем на $N$ ячеек длиной $dx = l/N$. В каждой ячейке — своё $L'dx$ (последовательно), $C'dx$ (параллельно), а также потери $R'dx$ и утечка $G'dx$.

Получается лестница из $N$ LC-звеньев. Это и есть дискретизация: производные по $x$ заменяются разностями.

## Шаг 2. Шахматная сетка

Напряжение живёт **на узлах** ($N+1$ точка), ток — **в серединах ячеек** ($N$ точек):

```
x:   U₀ ---- U₁ ---- U₂ ---- U₃          узлы, 801 штука
       I₀      I₁      I₂                середины ячеек, 800 штук
```

При таком расположении производные по $x$ считаются центральными разностями без интерполяции:

$$\frac{dU_i}{dt} = -\frac{I_i - I_{i-1}}{C'\,dx} - \frac{G'}{C'}\,U_i, \qquad
  \frac{dI_j}{dt} = -\frac{U_{j+1} - U_j}{L'\,dx} - \frac{R'}{L'}\,I_j$$

Вместе это $N+1 + N$ обыкновенных уравнений — одна система $\dot y = f(y,t)$, где $y$ — вектор из 1601 числа. Её и решает Рунге–Кутта.


In [ ]:
import numpy as np

# ---------------- параметры линии ----------------
Z0    = 50.0          # волновое сопротивление, Ом
v     = 2.0e8         # скорость волны, м/с
L_line = 143.0        # длина линии, м
beta  = 0.0           # затухание, 1/м (0 — без потерь)

# ---------------- источник и нагрузка ----------------
Rs    = 50.0          # внутреннее сопротивление, Ом
Rl    = np.inf        # нагрузка, Ом: np.inf = холостой ход
f_src = 7.0e6         # частота, Гц
amp   = 1.0           # амплитуда, В
T     = 5.0e-6        # сколько считаем, с

# ---------------- сетка ----------------
N  = 800
dx = L_line / N
dt = 0.4 * dx / v     # условие устойчивости, см. шаг 4

C = 1.0 / (Z0 * v)    # Ф/м
Lp = Z0 / v           # Гн/м
Rp = beta * Z0         # Ом/м   (условие Хевисайда)
Gp = beta / Z0        # См/м

Cdx, Ldx = C * dx, Lp * dx

print(f"λ = {v/f_src:.2f} м,  линия {L_line:.0f} м = {L_line/(v/f_src):.2f} λ")
print(f"τ = {L_line/v*1e9:.0f} нс (время пролёта)")
print(f"dx = {dx*1000:.0f} мм,  dt = {dt*1e12:.0f} пс,  шагов {int(T/dt)}")
print(f"за шаг волна проходит v·dt/dx = {v*dt/dx:.2f} ячейки")


## Шаг 3. Границы — здесь живёт вся физика

Крайние узлы **не интегрируются**: они каждый раз вычисляются из граничных условий.

* **Вход.** Источник с внутренним сопротивлением — это теорема Тевенена:
  $$U_0 = e(t) - R_s I_0$$
* **Выход.** Нагрузка задаёт связь напряжения и тока:
  $$U_N = R_l I_{N-1}, \qquad \text{при } R_l=\infty:\ I=0, \qquad \text{при } R_l=0:\ U_N=0$$

Функция `rhs` ниже возвращает производные для всех узлов и ячеек. Строки с `dU[0]` и `dU[-1]` равны нулю — эти значения подставляются из условий, а не интегрируются.


In [ ]:
def e(t):
    # напряжение источника, В
    return amp * np.sin(2 * np.pi * f_src * t)


def rhs(U, I, t, src=None, Rload=None):
    # производные по времени; src и Rload можно подменить (нужно для опытов ниже)
    src = e if src is None else src
    Rload = Rl if Rload is None else Rload

    dU = np.zeros_like(U)
    dI = -(U[1:] - U[:-1]) / Ldx - (Rp / Lp) * I              # dI/dt
    dU[1:-1] = -(I[1:] - I[:-1]) / Cdx - (Gp / C) * U[1:-1]   # dU/dt

    dU[0] = 0.0                                            # U₀ = src − Rs·I₀
    dU[-1] = 0.0 if np.isfinite(Rload) else I[-1] / Cdx    # при Rl = ∞ узел «свободный»
    return dU, dI


def step(U, I, t, src=None, Rload=None):
    # один шаг Рунге–Кутты + граничные условия
    src = e if src is None else src
    Rload = Rl if Rload is None else Rload

    a1, b1 = rhs(U, I, t, src, Rload)
    a2, b2 = rhs(U + dt/2*a1, I + dt/2*b1, t + dt/2, src, Rload)
    a3, b3 = rhs(U + dt/2*a2, I + dt/2*b2, t + dt/2, src, Rload)
    a4, b4 = rhs(U + dt*a3,   I + dt*b3,   t + dt,   src, Rload)

    U = U + dt/6 * (a1 + 2*a2 + 2*a3 + a4)
    I = I + dt/6 * (b1 + 2*b2 + 2*b3 + b4)
    t = t + dt

    U[0] = src(t) - Rs * I[0]                  # вход
    if np.isfinite(Rload):
        U[-1] = Rload * I[-1]                  # нагрузка
    else:
        U[-1] = U[-1]                          # свободный узел
    return U, I, t


U = np.zeros(N + 1)
I = np.zeros(N)
x = np.linspace(0.0, L_line, N + 1)

print("состояние:", U.shape, "+", I.shape, "=", U.size + I.size, "чисел")
print("в t = 0 всё нулевое, источник ещё не включён: e(0) =", e(0.0))


## Шаг 4. Почему шаг такой маленький

Схема явная: чтобы она была устойчива, волна не должна за один шаг перескакивать больше одной ячейки (условие Куранта–Фридрихса–Леви):

$$v\,dt < dx \quad\Longrightarrow\quad dt = 0.4\ \frac{dx}{v}$$

Коэффициент 0.4 — запас: при 1.0 схема на самой границе устойчивости и начинает звенеть. При $N = 800$ это $dt = 357$ пс, то есть 14 000 шагов на 5 мкс.

## Шаг 5. Рунге–Кутта 4-го порядка

На каждом шаге правая часть считается **четыре раза**:

```
k1 = f(y,           t)
k2 = f(y + dt/2·k1, t + dt/2)
k3 = f(y + dt/2·k2, t + dt/2)
k4 = f(y + dt·k3,   t + dt)

y ← y + dt/6 · (k1 + 2·k2 + 2·k3 + k4)
```

Один вызов `f` — это проход по всем 1601 уравнению. Ошибка на шаг ~$dt^5$, за 14 000 шагов набегает доли процента.

**Важно:** граничные условия (`U₀` и `U_N`) подставляются в конце каждого шага — иначе источник «отстаёт» на полшага.


In [ ]:
import time
t = 0.0
n_steps = int(T / dt)
t0 = time.time()

Umax = Imax = 0.0
for _ in range(n_steps):
    U, I, t = step(U, I, t)
    if t > 2 * L_line / v:                 # после установления
        Umax = max(Umax, float(np.abs(U).max()))
        Imax = max(Imax, float(np.abs(Z0 * I).max()))

print(f"посчитано за {time.time()-t0:.1f} с, шагов {n_steps}")
print(f"|U|max = {Umax:.3f} В      |Z₀·I|max = {Imax:.3f} В")


## Шаг 6. Откуда берётся отражение

**Нигде в коде отражённая волна не добавляется.** Она появляется сама — потому что падающая волна не удовлетворяет граничному условию на нагрузке.

Смотрим момент, когда волна дошла до конца. В последней ячейке $U = 1$ В, $I = 20$ мА — отношение 50 Ом, это волновое сопротивление. А нагрузка требует $U = R_l I$.

Пусть $R_l = 150$ Ом. При 1 В и 20 мА должно быть 3 В — не сходится. Значит, появляется волна, бегущая назад, с такими $U_r$ и $I_r$, чтобы сумма легла на прямую $U = R_l I$. Для обратной волны $U_r = -Z_0 I_r$ (ток течёт навстречу), поэтому:

```
U = 1 + U_r ,      I = 0.02 − U_r/50
150·(0.02 − U_r/50) = 1 + U_r
3 − 3·U_r = 1 + U_r        →       U_r = 0.5 В
```

Отражение 0.5 от 1 — ровно $\rho = \frac{R_l - Z_0}{R_l + Z_0} = +0.5$.

Проверим крайние случаи той же формулой:

| Нагрузка | Условие на конце | $U_r$ | Что видно |
|---|---|---|---|
| $R_l = 0$ | $U = 0$ | $-1$ | напряжение гаснет, ток удваивается до 40 мА |
| $R_l = 50$ | $U = 50 I$ | $0$ | отражения нет |
| $R_l = \infty$ | $I = 0$ | $+1$ | напряжение удваивается до 2 В, ток нуль |

В счётной схеме это выглядит так: узел на конце получает напряжение из граничного условия, и разность $(U_N - U_{N-1})$ в следующем шаге гонит ток в последней ячейке **в обратную сторону**. Этот обратный ток и есть отражённая волна — дальше она сама ползёт налево, ячейка за ячейкой.

Проверим численно: разложим поле на падающую и отражённую волны и посмотрим на их отношение.


In [ ]:
def ab(U, I):
    # волны A (вправо) и B (влево) — в серединах ячеек
    Uh = 0.5 * (U[:-1] + U[1:])
    return 0.5 * (Uh + Z0 * I), 0.5 * (Uh - Z0 * I)


print(f"{'Rl, Ом':>8} {'ρ теор':>7} {'|A|':>6} {'|B|':>6} {'|B|/|A|':>8}")
for Rl_test in (np.inf, 150.0, 50.0, 50/3, 0.0):
    u = np.zeros(N + 1)
    cur = np.zeros(N)
    tt = 0.0
    for _ in range(int(3e-6 / dt)):           # ~4 пролёта, чтобы установилось
        u, cur, tt = step(u, cur, tt, src=e, Rload=Rl_test)
    A, B = ab(u, cur)
    rho_th = 1.0 if not np.isfinite(Rl_test) else (Rl_test - Z0) / (Rl_test + Z0)
    print(f"{Rl_test:8.4g} {rho_th:+7.2f} {np.abs(A).max():6.3f} {np.abs(B).max():6.3f} "
          f"{np.abs(B).max()/np.abs(A).max():8.2f}")


## Как волна ползёт по ячейкам

Возьмём короткую линию и импульс — и напечатаем номер ячейки, где сейчас максимум напряжения. За шаг волна проходит 0.4 ячейки, за 2.5 шага — ровно одну.

Если всё сделано правильно, номер будет расти: 1, 1, 2, 2, 3, 3, 4… — и никаких «прыжков через пол-линии».


In [ ]:
N2 = 60                                    # короткая линия: 60 ячеек
U2 = np.zeros(N2 + 1)
I2 = np.zeros(N2)
t2 = 0.0

def e_pulse(tt):
    # короткий гауссов горб, включился на 5 нс
    return np.exp(-((tt - 5e-9) / 5e-9) ** 2)

print(f"dx = {dx*1000:.0f} мм,  dx/v = {dx/v*1e9:.1f} нс = {dx/v/dt:.1f} шага")
print(f"{'шаг':>5} {'t, нс':>7} {'пик U в ячейке':>15}")
for k in range(121):
    if k % 10 == 0:
        print(f"{k:5d} {t2*1e9:7.1f} {int(np.argmax(np.abs(U2))):15d}")
    U2, I2, t2 = step(U2, I2, t2, src=e_pulse, Rload=np.inf)


## Что подкрутить, чтобы проверить себя

| Изменить | Что должно измениться |
|---|---|
| `Rl = 50.0` | `\|B\|/\|A\|` → 0, отражения нет, `\|U\|max` ≈ 0.5 |
| `Rl = 0.0` | `\|B\|/\|A\|` → 1, `\|U\|` на конце нуль, ток удвоен |
| `beta = 0.01` | амплитуда падает к концу как $e^{-\beta x}$ |
| `N = 400` | тот же ответ с точностью ~1 % за вдвое меньшее время |
| `N = 200` | расхождение ~4 % — почувствуйте, за что платите |
| `f_src = 3.5e6` | на той же линии вдвое меньше волн |
| `T = 1e-6` | не хватит времени на установление — `\|U\|max` будет меньше |

**Где ошибаются чаще всего**

1. **Забыли граничные условия в конце шага.** Волна «упирается» и отражается не с тем знаком.
2. **Взяли dt слишком большим.** `dt = dx/v` — уже на грани; при `dt = 2·dx/v` счёт разваливается за десяток шагов (числа уходят в 1e150).
3. **Поставили U и I в одни и те же точки.** Тогда производные по x требуют интерполяции, и схема либо тормозит, либо звенит. Шахматка — не украшение.
4. **Сравнили частоту с шагом.** На период должно приходиться хотя бы десяток шагов: при 7 МГц и `dt = 357` пс это 400 шагов на период, с запасом.

**Дальше:** тот же расчёт с потерями, произвольным источником и анимацией — в файле `лаб23_волны_в_линии.ipynb`.
